**Programmer:** python_scripts (Abhijith Warrier)

**PYTHON SCRIPT TO UNDERSTAND PYTHON’S DESCRIPTOR PROTOCOL — THE MAGIC BEHIND @property, METHODS & ORMs. 🐍🧠**

Descriptors are one of the most powerful internals in Python.
Any object that defines one of these methods:

- `__get__(self, instance, owner)`
- `__set__(self, instance, value)`
- `__delete__(self, instance)`

is called a **descriptor**.

They control how attribute access works on classes —
and they power features like:
- `@property`
- methods & bound method behavior
- ORM fields (Django, SQLAlchemy)
- dataclasses auto-generated fields
- staticmethod / classmethod

This DeepCut breaks down the descriptor protocol using clear examples.

---

## 📦 Import Standard Library

In [1]:
import inspect

---

## 🧩 Snippet 1 — A descriptor implements __get__, __set__, or __delete__

This simplest example logs attribute access using custom descriptor methods.

In [2]:
class LoggedAttribute:
    def __init__(self, initial):
        self.value = initial

    def __get__(self, instance, owner):
        print("Accessing attribute...")
        return self.value

    def __set__(self, instance, value):
        print("Setting attribute...")
        self.value = value

class Demo:
    x = LoggedAttribute(10)

obj = Demo()
print(obj.x)     # calls __get__
obj.x = 20       # calls __set__
print(obj.x)

Accessing attribute...
10
Setting attribute...
Accessing attribute...
20


---

## 🧠 Snippet 2 — @property is just a descriptor

@property wraps a function into a descriptor that controls:
- getting a value
- optional setting (with @x.setter)

In [3]:
class Temperature:
    def __init__(self, celsius):
        self._c = celsius

    @property
    def celsius(self):        # behaves like __get__
        return self._c

    @celsius.setter
    def celsius(self, value): # behaves like __set__
        if value < -273.15:
            raise ValueError("Below absolute zero")
        self._c = value

t = Temperature(25)
print(t.celsius)
t.celsius = 30
print(t.celsius)

25
30


---

## 🔍 Snippet 3 — Functions become bound methods via descriptor behavior

Functions stored on classes automatically implement __get__.
This produces:
- **unbound functions** when accessed on class
- **bound methods** when accessed on instances

In [4]:
class Example:
    def greet(self):
        print("Hello from instance!")

# Unbound function
print(Example.greet)

# Bound method (self will be passed automatically)
e = Example()
print(e.greet)

# Call it
e.greet()

<function Example.greet at 0x10a4c9800>
<bound method Example.greet of <__main__.Example object at 0x10a4a8d70>>
Hello from instance!


---

## 🧱 Snippet 4 — staticmethod & classmethod wrap functions using descriptors

Their descriptor behavior decides:
- whether to pass `self`
- whether to pass `cls`

In [5]:
class Tools:
    @staticmethod
    def ping():
        return "Static call"

    @classmethod
    def identify(cls):
        return f"Called on {cls.__name__}"

print(Tools.ping())        # No instance or class passed
print(Tools.identify())    # cls passed automatically

Static call
Called on Tools


---

## 🏗️ Snippet 5 — A practical example: descriptor-based ORM-style field

This pattern is used in Django, SQLAlchemy, Pydantic, attrs, etc.

In [6]:
class Field:
    def __init__(self):
        self.private_name = None

    def __set_name__(self, owner, name):
        # called once when class is created
        self.private_name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, None)

    def __set__(self, instance, value):
        setattr(instance, self.private_name, value)

class User:
    name = Field()
    age = Field()

u = User()
u.name = "Sam"
u.age = 29

print(u.name, u.age)

Sam 29


---

## 🔧 Snippet 6 — __set_name__ gives descriptors knowledge of their attribute name

In [7]:
class Label:
    def __set_name__(self, owner, name):
        print(f"Binding descriptor to: {name}")

    def __get__(self, instance, owner):
        return "constant"

class Demo:
    x = Label()

Binding descriptor to: x


---

## ✅ One-liner Takeaway

**Descriptors power Python’s attribute system — enabling @property, methods, static/class methods, and ORM fields using the same elegant protocol.**

---